# MobileNetV2: Train, Evaluate, Confusion Matrix, Misclassified Images, ROC

This notebook fine-tunes **MobileNetV2** on your image dataset using PyTorch and Torchvision.
It mirrors the ResNet‑18 notebook's coverage and adds a few quality-of-life helpers.

**Includes:**
- Train/val/test pipeline (with automatic split if you only have one folder)
- MobileNetV2 (ImageNet weights), classifier head replaced for your classes
- Metrics: accuracy, loss curves
- **Confusion matrix**
- **Misclassified images gallery**
- **ROC curve (one-vs-rest)** for multi-class
- Model checkpointing (best val accuracy)
- Simple inference helper


In [ ]:
# %% [setup]
import os, random, time, copy, itertools
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ==== USER CONFIG ====
# Point DATA_ROOT to your dataset. Two options:
# (A) Folder with class subfolders; we'll split into train/val/test automatically (set AUTO_SPLIT=True)
# (B) Standard structure: DATA_ROOT/train, DATA_ROOT/val, DATA_ROOT/test each with class subfolders (set AUTO_SPLIT=False)

DATA_ROOT = Path('data')  # change to your dataset root
AUTO_SPLIT = True         # set to False if you already have train/val/test folders
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.15, 0.15  # used when AUTO_SPLIT=True

BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 10
LR = 1e-3
FREEZE_BACKBONE = False   # set True to train only the classifier head first
NUM_WORKERS = 2

print('Device:', DEVICE)


In [ ]:
# %% [data/transforms]
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


In [ ]:
# %% [data/loaders]
def build_dataloaders(root: Path):
    if AUTO_SPLIT:
        full_ds = datasets.ImageFolder(root=root, transform=train_tfms)
        n_total = len(full_ds)
        n_train = int(n_total * TRAIN_RATIO)
        n_val   = int(n_total * VAL_RATIO)
        n_test  = n_total - n_train - n_val
        train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test],
                                                 generator=torch.Generator().manual_seed(SEED))
        # apply eval tfms to val/test
        val_ds.dataset.transform = test_tfms
        test_ds.dataset.transform = test_tfms
    else:
        train_ds = datasets.ImageFolder(root / 'train', transform=train_tfms)
        val_ds   = datasets.ImageFolder(root / 'val',   transform=test_tfms)
        test_ds  = datasets.ImageFolder(root / 'test',  transform=test_tfms)

    class_names = train_ds.dataset.classes if hasattr(train_ds, 'dataset') else train_ds.classes
    num_classes = len(class_names)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    return (train_loader, val_loader, test_loader, class_names, num_classes)

train_loader, val_loader, test_loader, class_names, NUM_CLASSES = build_dataloaders(DATA_ROOT)
print('Classes:', class_names)
print('Train/Val/Test sizes:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))


In [ ]:
# %% [model]
from torchvision.models import MobileNet_V2_Weights

weights = MobileNet_V2_Weights.IMAGENET1K_V1
base = models.mobilenet_v2(weights=weights)

# Replace classifier head
in_feats = base.classifier[-1].in_features
base.classifier[-1] = nn.Linear(in_feats, NUM_CLASSES)

if FREEZE_BACKBONE:
    for name, param in base.features.named_parameters():
        param.requires_grad = False
    # classifier remains trainable

base = base.to(DEVICE)
print(base.classifier)


In [ ]:
# %% [train/utils]
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return running_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_logits.append(outputs.cpu())
        all_labels.append(labels.cpu())
    avg_loss = running_loss/total
    acc = correct/total
    logits = torch.cat(all_logits) if all_logits else torch.empty(0)
    labels = torch.cat(all_labels) if all_labels else torch.empty(0)
    return avg_loss, acc, logits, labels

def plot_curves(hist):
    epochs = range(1, len(hist['train_loss'])+1)
    plt.figure()
    plt.plot(epochs, hist['train_loss'], label='train_loss')
    plt.plot(epochs, hist['val_loss'], label='val_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend(); plt.show()

    plt.figure()
    plt.plot(epochs, hist['train_acc'], label='train_acc')
    plt.plot(epochs, hist['val_acc'], label='val_acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend(); plt.show()


In [ ]:
# %% [train/fit]
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([p for p in base.parameters() if p.requires_grad], lr=LR)
steps_per_epoch = max(1, len(train_loader))
scheduler = OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=steps_per_epoch, epochs=EPOCHS)

best_wts = copy.deepcopy(base.state_dict())
best_acc = 0.0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(base, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = evaluate(base, val_loader, criterion)
    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(val_acc)
    for _ in range(steps_per_epoch):
        # step scheduler each batch's worth to approximate OneCycle over the epoch
        try: scheduler.step()
        except: pass
    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(base.state_dict())
        torch.save(best_wts, 'mobilenetv2_best.pt')
    dt = time.time() - t0
    print(f"Epoch {epoch+1}/{EPOCHS} - {dt:.1f}s | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={val_loss:.4f} acc={val_acc:.4f}")

base.load_state_dict(best_wts)
plot_curves(history)
print('Best Val Acc:', best_acc)


In [ ]:
# %% [eval/test+plots]
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    all_paths = []
    softmax = nn.Softmax(dim=1)
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs).cpu()
        probs = softmax(logits)
        preds = logits.argmax(dim=1)
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_preds), torch.cat(all_labels)

probs, preds, labels = predict_loader(base, test_loader)
test_acc = (preds == labels).float().mean().item() if len(labels)>0 else float('nan')
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix
if len(labels) > 0:
    cm = confusion_matrix(labels.numpy(), preds.numpy(), labels=list(range(NUM_CLASSES)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    plt.figure()
    disp.plot(xticks_rotation=45, colorbar=False)
    plt.title('Confusion Matrix - Test')
    plt.show()

# Misclassified images (sample up to 16)
# To show images, we need access to underlying dataset paths; handle both split modes.
def get_item_with_path(ds, idx):
    if hasattr(ds, 'dataset') and hasattr(ds.dataset, 'imgs'):
        # Subset of ImageFolder (from random_split)
        return ds[idx][0], ds[idx][1], ds.dataset.imgs[ds.indices[idx]][0]
    elif hasattr(ds, 'dataset') and hasattr(ds.dataset, 'samples'):
        return ds[idx][0], ds[idx][1], ds.dataset.samples[ds.indices[idx]][0]
    elif hasattr(ds, 'samples'):
        return ds[idx][0], ds[idx][1], ds.samples[idx][0]
    else:
        return ds[idx][0], ds[idx][1], None

mis_idx = torch.nonzero(preds != labels, as_tuple=False).flatten().tolist()
print(f"Misclassified: {len(mis_idx)}")
n_show = min(16, len(mis_idx))

if n_show > 0:
    # re-fetch items with paths from original dataset in order
    # we rely on test_loader.dataset
    import math
    rows = math.ceil(n_show/4)
    plt.figure(figsize=(12, 3*rows))
    for i in range(n_show):
        idx = mis_idx[i]
        img_t, true_y = test_loader.dataset[idx]
        # unnormalize for display
        img_np = img_t.numpy().transpose(1,2,0) * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
        img_np = np.clip(img_np, 0, 1)
        pred_y = preds[idx].item()
        plt.subplot(rows, 4, i+1)
        plt.imshow(img_np)
        plt.axis('off')
        plt.title(f"pred: {class_names[pred_y]}\ntrue: {class_names[true_y]}")
    plt.suptitle('Misclassified Examples')
    plt.tight_layout()
    plt.show()

# ROC (one-vs-rest)
if NUM_CLASSES >= 2 and len(labels) > 0:
    y_true = labels.numpy()
    y_score = probs.numpy()
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    if NUM_CLASSES == 2:
        fpr, tpr, _ = roc_curve(y_bin[:,1], y_score[:,1])
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
        plt.plot([0,1],[0,1],'--')
        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend(); plt.show()
    else:
        # micro-average
        fpr, tpr, _ = roc_curve(y_bin.ravel(), y_score.ravel())
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, label=f'Micro-average AUC = {roc_auc:.3f}')
        plt.plot([0,1],[0,1],'--')
        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve (OvR)'); plt.legend(); plt.show()


In [ ]:
# %% [inference/helper]
from PIL import Image

def predict_image(path: str):
    img = Image.open(path).convert('RGB')
    t = test_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = base(t)
        probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
        pred_idx = int(probs.argmax())
    return class_names[pred_idx], probs[pred_idx], {cls: float(p) for cls, p in zip(class_names, probs)}

# Example:
# pred, conf, dist = predict_image('some_image.jpg')
# print(pred, conf)
